In [8]:
# ---------------- PART 4 ----------------

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ---------------- LOAD DATA ----------------

df = pd.read_csv("students.csv")

print("\n--- HEAD ---")
print(df.head())

print("\n--- SHAPE ---")
print(df.shape)

print("\n--- DTYPES ---")
print(df.dtypes)

print("\n--- DESCRIBE ---")
print(df.describe())

print("\n--- PASS/FAIL COUNT ---")
print(df['passed'].value_counts())

# ---------------- SUBJECT ANALYSIS ----------------

subject_cols = ['math', 'science', 'english', 'history', 'pe']

print("\n--- AVG SCORES (PASS) ---")
print(df[df['passed']==1][subject_cols].mean())

print("\n--- AVG SCORES (FAIL) ---")
print(df[df['passed']==0][subject_cols].mean())

df["avg_score"] = df[subject_cols].mean(axis=1)

topper = df.loc[df["avg_score"].idxmax()]
print("\n--- TOPPER ---")
print(topper["name"], topper["avg_score"])

# ---------------- MATPLOTLIB ----------------

# 1 BAR CHART
plt.figure()
df[subject_cols].mean().plot(kind='bar')
plt.title("Average Score per Subject")
plt.xlabel("Subjects")
plt.ylabel("Score")
plt.savefig("plot1_bar.png")
plt.show()

# 2 HISTOGRAM
plt.figure()
plt.hist(df['math'], bins=5)
mean_math = df['math'].mean()
plt.axvline(mean_math, linestyle='dashed')
plt.title("Math Score Distribution")
plt.xlabel("Math Score")
plt.ylabel("Frequency")
plt.savefig("plot2_hist.png")
plt.show()

# 3 SCATTER
plt.figure()
pass_df = df[df['passed']==1]
fail_df = df[df['passed']==0]

plt.scatter(pass_df['study_hours_per_day'], pass_df['avg_score'], label='Pass')
plt.scatter(fail_df['study_hours_per_day'], fail_df['avg_score'], label='Fail')

plt.xlabel("Study Hours")
plt.ylabel("Avg Score")
plt.title("Study Hours vs Avg Score")
plt.legend()
plt.savefig("plot3_scatter.png")
plt.show()

# 4 BOXPLOT
plt.figure()
pass_att = pass_df['attendance_pct'].tolist()
fail_att = fail_df['attendance_pct'].tolist()

plt.boxplot([pass_att, fail_att], labels=['Pass', 'Fail'])
plt.title("Attendance Distribution")
plt.savefig("plot4_box.png")
plt.show()

# 5 LINE
plt.figure()
plt.plot(df['name'], df['math'], marker='o', label='Math')
plt.plot(df['name'], df['science'], marker='x', label='Science')

plt.xticks(rotation=45)
plt.title("Math vs Science Scores")
plt.legend()
plt.savefig("plot5_line.png")
plt.show()

# ---------------- SEABORN ----------------

# 6 BARPLOT
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
sns.barplot(data=df, x='passed', y='math')
plt.title("Math vs Pass/Fail")

plt.subplot(1,2,2)
sns.barplot(data=df, x='passed', y='science')
plt.title("Science vs Pass/Fail")

plt.savefig("plot6_seaborn_bar.png")
plt.show()

# 7 SCATTER + REG
plt.figure()

sns.scatterplot(data=df, x='attendance_pct', y='avg_score', hue='passed')
sns.regplot(data=pass_df, x='attendance_pct', y='avg_score', scatter=False)
sns.regplot(data=fail_df, x='attendance_pct', y='avg_score', scatter=False)

plt.title("Attendance vs Avg Score")
plt.savefig("plot7_seaborn_scatter.png")
plt.show()

# comment:
# seaborn felt easier for styling and grouping,
# while matplotlib required more manual control

# ---------------- MACHINE LEARNING ----------------

X = df[['math','science','english','history','pe','attendance_pct','study_hours_per_day']]
y = df['passed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("\nTraining Accuracy:", model.score(X_train_scaled, y_train))
print("Test Accuracy:", model.score(X_test_scaled, y_test))

preds = model.predict(X_test_scaled)

print("\nPredictions:")
for i in range(len(preds)):
    name = df.loc[X_test.index[i], 'name']
    actual = y_test.iloc[i]
    pred = preds[i]
    result = "Correct" if actual == pred else "Wrong"
    print(name, actual, pred, result)

# FEATURE IMPORTANCE

coef = model.coef_[0]
features = X.columns

feature_importance = list(zip(features, coef))
feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)

print("\nFeature Importance:")
for f, c in feature_importance:
    print(f, c)

plt.figure()
colors = ['green' if c > 0 else 'red' for c in coef]

plt.barh(features, coef, color=colors)
plt.title("Feature Importance")
plt.savefig("plot8_feature_importance.png")
plt.show()

# BONUS

new_student = [[75,70,68,65,80,82,3.2]]
new_scaled = scaler.transform(new_student)

prediction = model.predict(new_scaled)[0]
prob = model.predict_proba(new_scaled)

print("\nNew Student Prediction:", "Pass" if prediction==1 else "Fail")
print("Probability:", prob)

EmptyDataError: No columns to parse from file